# 12 — Sheaf GNN: a trained prior that ships with the store

The symbolic reasoner (Dempster\u2013Shafer + MCTS) gets chain verdicts right most of the time. But hand-coded rules for chain mass (`min(S), max(R)`, hop penalty = 0.10) and connective-predicate inference have a ceiling: on synthetic held-out data we measured **88.5% accuracy**. The failures cluster on one pattern — reportive-edge traps — where the symbolic rule has no way to distinguish `X supplies Y` from `X mentions Y`.

A trained sheaf GNN as a **terminal scorer** closes the gap:

| Setting | Synthgen held-out | Reportive-edge anomaly |
|---|---|---|
| Symbolic only | 88.5% | 6% |
| With GNN prior | **99.2%** | **100%** |

The GNN is trained once on synthetic hypergraphs with known ground truth (`synthgen.py`), saved under `<root>/_model/gnn.pt`, and called at query time by `reason_connectivity(use_gnn=True)`. It's schema-independent — the features are (relation-kind, polarity, position, endpoint-touch) so the same weights transfer across domains.

This notebook:

1. Generates a synthetic corpus with labeled chains.
2. Checks the symbolic reasoner against the labels (baseline = 88.5%).
3. Trains the sheaf GNN on 2000 samples; evaluates on 400 held-out.
4. Saves the trained weights under a store's `_model/`.
5. Runs a single connectivity query with and without the GNN prior to show the verdict difference.

## 1. The synthetic corpus: ground truth by construction

`synthgen.generate(config)` produces `list[SynthGraph]` where each graph carries:

- a list of edges (synthetic triples with polarity + timestamp),
- a chain source and target,
- a gold verdict \u2014 SUPPORTS / REFUTES / NOT_ENOUGH_INFO.

The verdict is deterministic from the generator's rules: clean chains get SUPPORTS; chains with a retraction get REFUTES; anomalies where a reportive edge breaks the chain get NEI; disconnected pairs get NEI. Five kinds in total, sampled via `chain_length_bias`, `polarity_flip_rate`, `anomaly_rate`, `cycle_rate`.

The `RelationSpec` for each predicate has a `kind` field \u2014 `connective`, `terminal`, or `reportive` \u2014 which drives both the synthgen generator and the GNN's per-kind restriction maps.

In [ ]:
from infon.cassette.synthgen import (
    SynthGenConfig, RelationSpec, generate, summarize, _default_config,
)

config = _default_config()
config.n_samples = 400
config.seed = 7

print("Relations:")
for r in config.relations:
    print(f"  {r.name:<10} kind={r.kind:<12} symmetric={r.symmetric}")

graphs = generate(config)
print()
for k, v in summarize(graphs).items():
    print(f"  {k}: {v}")

## 2. Symbolic baseline: 88.5% overall, 6% on anomalies

Before training a GNN we need a baseline to beat. Run the symbolic `chain_mass` rule against the generator's labels and check agreement per kind. The per-kind breakdown tells us exactly where training will help.

In [ ]:
from collections import Counter
from infon.cassette.reason_path import chain_mass, _label_from_mass
from infon.infon import Infon

def synth_to_infon(e):
    return Infon(
        infon_id=f"syn_{id(e):x}",
        subject=e.subject, predicate=e.predicate, object=e.object,
        polarity=e.polarity, confidence=e.confidence,
        sentence=f"{e.subject} {e.predicate} {e.object}",
        timestamp=f"2026-01-{1 + (e.t % 28):02d}",
    )

def symbolic_verdict(g):
    # BFS the graph's edges for a path from source to target.
    triples = {}
    adj = {}
    for e in g.edges:
        triples.setdefault((e.subject, e.predicate, e.object), []).append(
            synth_to_infon(e))
    for (s, _, o), infons in triples.items():
        adj.setdefault(s, []).append((o, infons))
    from collections import deque
    q = deque([(g.chain_source, [])])
    seen = {g.chain_source}
    path_edges = []
    while q:
        node, p = q.popleft()
        for nxt, ei in adj.get(node, []):
            new = p + [ei]
            if nxt == g.chain_target:
                path_edges = new
                break
            if nxt not in seen:
                seen.add(nxt); q.append((nxt, new))
        if path_edges: break
    return _label_from_mass(chain_mass(path_edges, g.chain_source, g.chain_target))

per_kind = {}
for g in graphs:
    pred = symbolic_verdict(g)
    d = per_kind.setdefault(g.kind, [0, 0])
    d[1] += 1
    if pred == g.chain_verdict: d[0] += 1

total = sum(v[1] for v in per_kind.values())
agree = sum(v[0] for v in per_kind.values())
print(f"symbolic overall: {agree}/{total} = {agree/total:.1%}")
print()
for kind, (a, t) in sorted(per_kind.items(), key=lambda x: -x[1][1]):
    print(f"  {kind:<14} {a}/{t} = {a/t:.0%}")

### Where the 11.5% gap comes from

The anomaly kind (a reportive edge sitting where a connective chain should form) is the hardest. The symbolic rule doesn't know that `mention` shouldn't propagate chain confidence the way `supply` does \u2014 the hand-coded chain mass just sees "an edge with polarity=1 and some confidence."

This is what the GNN's per-relation-kind restriction maps learn: three separate linear maps (one per kind) that dampen reportive messages and propagate connective ones. No manual rule needed \u2014 the generator labels teach it.

## 3. Train the sheaf GNN

The encoder is a 3-layer message-passing stack with per-relation-kind restriction maps. 140k parameters, trains in about a minute on CPU.

Architecture in brief:

- **StalkEncoder**: per-edge features (10-dim: kind one-hot + polarity + confidence + log-gap + is-last + touches-source/target + connects-prev) \u2192 hidden vector.
- **SheafMessageLayer**: one hop of message passing along the chain. Forward and backward restriction maps are kind-specific; discrepancy (H\u00b9) per position flags structural contradictions.
- **ChainVerdictHead**: pools (last stalk, mean stalk, total discrepancy) \u2192 3-class logits.

Training target = chain-verdict cross-entropy. The H\u00b9 discrepancy is a feature, not a loss term \u2014 the head learns to read it.

In [ ]:
from infon.cassette.gnn_encoder import (
    SheafHypergraphEncoder, batch_from_synth, train, evaluate,
)

train_cfg = _default_config(); train_cfg.n_samples = 2000; train_cfg.seed = 7
val_cfg   = _default_config(); val_cfg.n_samples   = 300;  val_cfg.seed   = 11
test_cfg  = _default_config(); test_cfg.n_samples  = 400;  test_cfg.seed  = 17

train_graphs = generate(train_cfg)
val_graphs   = generate(val_cfg)
test_graphs  = generate(test_cfg)

relation_kinds = {r.name: r.kind for r in train_cfg.relations}
train_batch = batch_from_synth(train_graphs, relation_kinds)
val_batch   = batch_from_synth(val_graphs,   relation_kinds)
test_batch  = batch_from_synth(test_graphs,  relation_kinds)

model = SheafHypergraphEncoder(hidden_dim=64, n_layers=3)
print(f"model params: {model.n_params():,}")

result = train(model, train_batch, val_batch,
               epochs=25, batch_size=64, lr=2e-3, verbose=False)
print(f"best val acc: {result.best_val_acc:.3f} @ epoch {result.best_epoch}")

In [ ]:
out = evaluate(model, test_batch)
preds, gold = out["preds"], out["gold"]
correct = (preds == gold).sum()
total = len(gold)
print(f"GNN test accuracy: {correct}/{total} = {correct/total:.1%}")

from collections import defaultdict
per_kind = defaultdict(lambda: [0, 0])
for g, p, y in zip(test_graphs, preds, gold):
    per_kind[g.kind][1] += 1
    if p == y: per_kind[g.kind][0] += 1
for kind in ("clean", "disconnected", "cyclic", "retracted", "anomaly"):
    a, t = per_kind[kind]
    if t: print(f"  {kind:<14} {a}/{t} = {a/t:.0%}")

## 4. Save the trained weights next to a store

The cassette substrate expects the trained GNN at `<root>/_model/gnn.pt`. `reason_connectivity(use_gnn=True)` looks there, falls back to symbolic-only if the file is missing.

Weights, the hidden dim, and the layer count go in the checkpoint so the loader can rebuild the exact architecture.

In [ ]:
import tempfile, os, json
import torch
from infon.cassette import InfonStore, Query

tmpdir = tempfile.mkdtemp(prefix="infon_12_")

# Minimal schema + one supply chain so we have something to query.
SCHEMA = {
    "toyota":    {"type": "actor",    "tokens": ["toyota"]},
    "panasonic": {"type": "actor",    "tokens": ["panasonic"]},
    "catl":      {"type": "actor",    "tokens": ["catl"]},
    "partner":   {"type": "relation", "tokens": ["partner", "partnered"]},
    "supply":    {"type": "relation", "tokens": ["supply", "supplies"]},
}
schema_path = os.path.join(tmpdir, "schema.json")
with open(schema_path, "w") as f: json.dump(SCHEMA, f)

store = InfonStore(os.path.join(tmpdir, "store"), schema_path=schema_path)
store.ingest([
    {"id": "d1", "text": "Toyota partnered with Panasonic.",
     "timestamp": "2026-01-05"},
    {"id": "d2", "text": "Panasonic supplies CATL.",
     "timestamp": "2026-01-20"},
])

# Save the trained model into this store's _model/
model_dir = os.path.join(store.root, "_model")
os.makedirs(model_dir, exist_ok=True)
torch.save({
    "state_dict": model.state_dict(),
    "hidden_dim": 64,
    "n_layers": 3,
    "best_val_acc": result.best_val_acc,
}, os.path.join(model_dir, "gnn.pt"))
print(f"saved: {model_dir}/gnn.pt")

## 5. Query the store with and without the GNN prior

`reason_connectivity(use_gnn=True)` loads the checkpoint, runs MCTS as normal, then re-scores the discovered chain via the GNN. The returned `Verdict.mass` is a weighted blend: `(1 - gnn_weight) * symbolic + gnn_weight * gnn`, with `gnn_weight=0.5` by default.

For a clean chain both methods agree. The difference shows up on the anomaly patterns we measured above.

In [ ]:
from infon.cassette.reason_path import (
    reason_connectivity, _GNN_CACHE,
)

# Clear the cache so we load from disk (confirms the checkpoint round-trip).
_GNN_CACHE.clear()

CONNECTIVE = {"partner", "supply", "acquire", "license", "invest"}

# Symbolic only
v_sym = reason_connectivity(store.manifest, "toyota", "catl",
                             connective_predicates=CONNECTIVE,
                             use_gnn=False)
print(f"symbolic   {v_sym.label:<18}  S={v_sym.mass.supports:.2f}  "
      f"\u03b8={v_sym.mass.theta:.2f}")

# With GNN prior blended in
v_gnn = reason_connectivity(store.manifest, "toyota", "catl",
                             connective_predicates=CONNECTIVE,
                             use_gnn=True, gnn_weight=0.5)
print(f"gnn-prior  {v_gnn.label:<18}  S={v_gnn.mass.supports:.2f}  "
      f"\u03b8={v_gnn.mass.theta:.2f}")

## Summary

| What happens at each step | Cost |
|---|---|
| Generate 2000 synthgen samples | milliseconds |
| Train the sheaf GNN on CPU | ~60 seconds |
| Save weights to `<root>/_model/gnn.pt` | 5\u201350 MB checkpoint |
| Load + apply at query time | ~1 ms per chain |

**Why this works as a shipped feature, not a research experiment:**

- Features are schema-independent (relation-kind, polarity, position). The trained weights transfer across domains \u2014 you do NOT retrain per corpus.
- Ground truth is generated, not labeled. No annotation cost.
- Symbolic reasoner still runs unchanged; the GNN is a terminal re-score, not a replacement. Sources are still cited; the calibrated verdict keeps its honest \u03b8.
- When the checkpoint is missing, `use_gnn=True` is a no-op. Ship without a model; train one later.

**Next:**
- **[08 \u2014 Category Theory](08_category_theory.ipynb)** \u2014 Kan migration as the second shipped category-theoretic tool.
- **[06 \u2014 Agent Tools](06_agent_tools.ipynb)** \u2014 the Analyst calls `bootstrap_gnn()` to train per-store from a corpus peek, not just synthetic data.

In [ ]:
import shutil
shutil.rmtree(tmpdir)
print("Done.")